<div align="center">

# MAPD-B Final Project: Mini-Batch K-Means

**Kevin Brugnera · Marco Lorenzato · Riccardo Ferrante · Federico Scianna**

ID: 2196578 · 2197529 · 2196576 · 2183435

</div>

---

## Introduction

Kmeans vs MiniBatch ..

This notebook documents our implementation of Mini‑Batch K‑Means using Apache Spark deployed on the CloudVeneto cluster. It includes cluster configuration and node topology, data preparation and distribution, a distributed Mini‑Batch K‑Means implementation, benchmarking methodology and results across different node configurations, and diagnostic performance analyses.

## 1. Cluster Setup



## 2. RCV1 Dataset

RCV1..

### 2.1 Dataset Exploration

papers .. labels ..

### 2.2 Dataset Generation and Distribution

file data generation ...

## 3. Implementation and Workflow

We present here our code structure and execution flow.

### 3.1 Implementation of clustering algorithms

**Note:** All the functions described in this notebook can be consulted in the file **functions.py**. Not all the functions that appear in the file have been illustrated here. We just focused on the most important ones, which are crucial to understand the workflow and the core content of the project.

Here we describe the Spark-parallelized implementation of both the classic and the Mini-batch versions of $k$-Means algorithm.

We first implemented the classic version of the algorithm in the `classic_kmeans` function reported below. In this function, we start with the `.takeSample` action to initialize `k` centroids by drawing `k` random points from the dataset. During each step of the following `for` loop, first the centroids are broadcast to the Executors. Then, the map-reduce transformations `.map` and `.reduceByKey` are used to assign each point to the closest cluster(this makes use of the custom function `closest_idx`) and compute the vector sums and point count needed to obtain the new centroids. Finally, these are returned to the Driver with the `.collect()` action, where the new centroids are eventually computed. The old broadcast variable is erased from the Workers' memory using `.destroy()`.

```python
def classic_kmeans(rdd_train, k: int, epochs: int, seed: int):
    """Executes the standard K-Means algorithm using Spark."""

    # Initialize centers randomly from training set
    centers = rdd_train.takeSample(False, k, seed)
    
    for _ in range(epochs):
        #Broadcast
        centers_np = np.array(centers)
        bc_centers = rdd_train.context.broadcast(centers_np)
        
        # Map using the broadcasted variable, returns -> (center_idx, (point,count)) of course count will always be 1
        mapped_points = rdd_train.map(lambda x: (closest_idx(x, bc_centers.value), (x, 1)))
        
        # Reduce by key (center_idx) and gives -> (center_idx, (vectorial_sum, population))
        reduced_points = mapped_points.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
        
        # Update centers and gives them to master -> (center_idc, vectorial_sum/population)
        new_centers_rdd = reduced_points.map(lambda x: (x[0], x[1][0] / x[1][1]))
        new_centers_dict = dict(new_centers_rdd.collect())
        
        # Update centers for next iteration
        centers = [new_centers_dict.get(i, centers[i]) for i in range(k)]
        bc_centers.destroy()

Then, we implemented the Mini-batch version. This approach employs small batches to optimize the classic algorithm. We started from the version proposed by D. Sculley (https://dl.acm.org/doi/10.1145/1772690.1772862), whose pseudocode is reported below. This introduces a gradient-descent approach, introducing of a learning rate term $\eta$ that is updated point-by-point at each step during the inner `for` loop over the batch.

**Algorithm 1:** Mini-batch $k$-Means.
***
1. **Given:** $k$, mini-batch size $b$, iterations $t$, data set $X$
2. Initialize each $\mathbf{c} \in C$ with an $\mathbf{x}$ picked randomly from $X$
3. $\mathbf{v} \leftarrow 0$
4. **for** $i = 1$ to $t$ **do**
5. &nbsp;&nbsp;&nbsp;&nbsp; $M \leftarrow b$ examples picked randomly from $X$
6. &nbsp;&nbsp;&nbsp;&nbsp; **for** $\mathbf{x} \in M$ **do**
7. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $\mathbf{d}[\mathbf{x}] \leftarrow f(C, \mathbf{x})$ &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; *// Cache the center nearest to $\mathbf{x}$*
8. &nbsp;&nbsp;&nbsp;&nbsp; **end for**
9. &nbsp;&nbsp;&nbsp;&nbsp; **for** $\mathbf{x} \in M$ **do**
10. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $\mathbf{c} \leftarrow \mathbf{d}[\mathbf{x}]$ &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; *// Get cached center for this $\mathbf{x}$*
11. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $\mathbf{v}[\mathbf{c}] \leftarrow \mathbf{v}[\mathbf{c}] + 1$ &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; *// Update per-center counts*
12. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $\eta \leftarrow \frac{1}{\mathbf{v}[\mathbf{c}]}$ &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; *// Get per-center learning rate*
13. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $\mathbf{c} \leftarrow (1 - \eta)\mathbf{c} + \eta\mathbf{x}$ &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; *// Take gradient step*
14. &nbsp;&nbsp;&nbsp;&nbsp; **end for**
15. **end for**
***

We then implemented the Spark-parallelized version of the Mini-batch $k$-Means algorithm. First, the centroids are initialized randomly using the `.takeSample` action, and the historical count array `v` is initialized to zero. The `fraction` parameter, quantifying the percentage of samples per batch, is then computed employing the `.count()` action. During the following `for` loop, the centroids are broadcast to the Executors and a random mini-batch is drawn from the dataset using the `.sample()` transformation. Map-reduce transformations(`.map` and .`reduceByKey`) are then employed to assign the points to the nearest centroid, and to compute the vector sums and point counts for each cluster. These are then returned to the Driver via the `.collect()` action. Finally, the centroids are computed with a gradient step: for each cluster, the historical volume `v` is updated and the learning rate `$eta$` is calculated. The centroid is moved towards the center of the current batch according to this learning rate. Lastly, the broadcast variable is then destroyed to free the memory.

```python
def minibatch_kmeans(rdd_train, k: int, b: int, epochs: int, seed: int):
    """Executes the Mini-Batch K-Means algorithm using Spark."""
    centers_list = rdd_train.takeSample(False, k, seed)
    centers = np.array(centers_list) 
    
    v = np.zeros(k)
    
    # Calculate fraction for Spark's .sample()
    total_count = rdd_train.count()
    fraction = min(1, float(b)/total_count)
    
    for epoch in range(epochs):
        # Broadcast the centers
        bc_centers = rdd_train.context.broadcast(centers)
        
        # Distributed sampling on workers
        rdd_batch = rdd_train.sample(False, fraction)
        
        # Distributed mapping --> (cluster_idx: (point,1))
        mapped_batch = rdd_batch.map(
            lambda x: (closest_idx(x, bc_centers.value), (x, 1))
        )
        
        # Distributed reduction, collecting only the k aggregates
        reduced_batch = mapped_batch.reduceByKey(
            lambda a, b: (a[0] + b[0], a[1] + b[1])
        ).collect()
        
        # Local update on the Driver 
        for c_idx, (sum_x, count) in reduced_batch:
            v[c_idx] += count               
            eta = count / v[c_idx]          
            batch_mean = sum_x / count      
            centers[c_idx] = (centers[c_idx] * (1.0 - eta)) + (batch_mean * eta)
            
        bc_centers.destroy()
        
    return centers.tolist()

## 4. Results and Benchmarks

---

#### References

- Sculley, D. (2010). *Web-scale k-means clustering*.
- MAPD-B UniPD course lecture notes.
- Scikit-learn RCV1 dataset: https://scikit-learn.org/0.18/datasets/rcv1.html
- Lewis, D. D., Yang, Y., Rose, T. G., & Li, F. (2004). *RCV1: A new benchmark collection for text categorization research*. Journal of Machine Learning Research, 5, 361-397.
- Apache Spark Parquet documentation: https://spark.apache.org/docs/latest/sql-data-sources-parquet.html
- Scikit-learn TruncatedSVD: https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html